In [1]:
import pandas as pd

df = pd.read_csv("/marketing_AB.csv")
print(df.head())

   Unnamed: 0  user id test group  converted  total ads most ads day  \
0           0  1069124         ad      False        130       Monday   
1           1  1119715         ad      False         93      Tuesday   
2           2  1144181         ad      False         21      Tuesday   
3           3  1435133         ad      False        355      Tuesday   
4           4  1015700         ad      False        276       Friday   

   most ads hour  
0             20  
1             22  
2             18  
3             10  
4             14  


In [2]:
print(df.shape)
print(df["test group"].value_counts())

(588101, 7)
test group
ad     564577
psa     23524
Name: count, dtype: int64


In [3]:
rates = df.groupby("test group")["converted"].mean()
print(rates * 100)

test group
ad     2.554656
psa    1.785411
Name: converted, dtype: float64


In [6]:
ad_rate = rates["ad"]
psa_rate = rates["psa"]

abs_lift = (ad_rate - psa_rate) * 100
rel_lift = (ad_rate - psa_rate) / psa_rate * 100

print("Absolute lift (percentage points):", round(abs_lift, 3))
print("Relative lift (%):", round(rel_lift, 2))

Absolute lift (percentage points): 0.769
Relative lift (%): 43.09


In [7]:
counts = df.groupby("test group")["converted"].agg(["sum", "count"])
print(counts)

              sum   count
test group               
ad          14423  564577
psa           420   23524


In [8]:
from scipy import stats
import numpy as np

x_ad = counts.loc["ad", "sum"]
n_ad = counts.loc["ad", "count"]
x_psa = counts.loc["psa", "sum"]
n_psa = counts.loc["psa", "count"]

p_ad = x_ad / n_ad
p_psa = x_psa / n_psa
p_pool = (x_ad + x_psa) / (n_ad + n_psa)

se = np.sqrt(p_pool * (1 - p_pool) * (1 / n_ad + 1 / n_psa))
z = (p_ad - p_psa) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print("z-statistic:", round(z, 3))
print("p-value:", p_value)

z-statistic: 7.37
p-value: 1.7053025658242404e-13


In [9]:
diff = p_ad - p_psa

se_diff = np.sqrt(p_ad * (1 - p_ad) / n_ad + p_psa * (1 - p_psa) / n_psa)

z_crit = stats.norm.ppf(0.975)

ci_low = diff - z_crit * se_diff
ci_high = diff + z_crit * se_diff

print("Difference:", round(diff * 100, 3), "percentage points")
print("95% CI: [", round(ci_low * 100, 3), ",", round(ci_high * 100, 3), "] percentage points")

Difference: 0.769 percentage points
95% CI: [ 0.595 , 0.943 ] percentage points


## Conclusion

- Ad group conversion: 2.55% vs control (PSA): 1.79%
- Lift: +0.77 percentage points (43% relative lift)
- p-value < 0.001, 95% CI: [0.60, 0.94] percentage points
- **Recommendation: Launch.** The lift is statistically significant and the whole confidence interval is above zero.

## Conclusion

- Ad group conversion: 2.55% vs control (PSA): 1.79%
- Lift: +0.77 percentage points (43% relative lift)
- p-value < 0.001, 95% CI: [0.60, 0.94] percentage points
- **Recommendation: Launch.** The lift is statistically significant and the whole confidence interval is above zero.